# HalluGuard-Qwen3-4B — LLM-AggreFact DEV smoke test

This notebook is an engineering smoke test only. It evaluates a small, stratified sample from the **DEV** split and must not be used to report a final benchmark result or tune decoding based on test labels. Version 1 remains immutable.

Model: `lrsbrgrn/HalluGuard-Qwen3-4B`. The prompt and XML response protocol below follow the [official model card](https://huggingface.co/lrsbrgrn/HalluGuard-Qwen3-4B/blob/main/README.md); the research reference is the [ACL 2026 paper](https://aclanthology.org/2026.findings-acl.835/). HalluGuard is not routed through `MiniCheck.score()`.

## 1. Setup and credentials

Enable Internet and **2x T4 GPU** in Kaggle. If Hugging Face access is required, add `HF_TOKEN` through Kaggle Secrets (preferred) or the environment; never paste a token into this notebook.

In [1]:
!pip install -q -U 'transformers>=4.51.0' accelerate datasets pandas scikit-learn

import gc
import json
import os
import re
import time
from pathlib import Path

import pandas as pd
import torch
from datasets import load_dataset
from sklearn.metrics import balanced_accuracy_score
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = 'lrsbrgrn/HalluGuard-Qwen3-4B'
DATASET_NAME = 'lytang/LLM-AggreFact'
SPLIT = 'dev'  # Guardrail: this smoke test must never use test.
SMOKE_PER_DATASET_LABEL = 2
OUTPUT_DIR = Path('/kaggle/working/halluguard_dev_smoke')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_PATH = OUTPUT_DIR / 'predictions_checkpoint.csv'

assert SPLIT == 'dev', 'This notebook is DEV smoke-test only; do not switch to test in Step 2.'
assert torch.cuda.is_available(), 'Enable a Kaggle GPU accelerator.'
assert torch.cuda.device_count() >= 2, 'Configure Kaggle for 2x T4 before running this smoke test.'

def get_hf_token():
    # Kaggle Secrets takes precedence; errors intentionally reveal no secret value.
    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret('HF_TOKEN')
        if token:
            return token
    except Exception:
        pass
    return os.environ.get('HF_TOKEN')

HF_TOKEN = get_hf_token()
print(f'GPUs available: {torch.cuda.device_count()}')
for gpu_id in range(torch.cuda.device_count()):
    print(f'GPU {gpu_id}: {torch.cuda.get_device_name(gpu_id)}')
print('Hugging Face credential source checked (value not displayed).')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 82.7 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 16.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 21.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 95.9 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 88.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 19.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 64.3 MB/s eta 0:00:00:00:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
ydata-profiling 4.18.4 requires pandas!=1.4.0,<3.0,>1.5, b

## 2. Load the DEV-only stratified smoke sample

The sample is stratified by dataset and binary label solely to exercise the pipeline across the available DEV data. It does not select generation settings or provide a final model comparison. `source_index` is retained for checkpoint/resume auditing.

In [3]:
dataset_dict = load_dataset(DATASET_NAME, token=HF_TOKEN)
assert SPLIT in dataset_dict, f'Missing required DEV split: {dataset_dict}'
dev_df = pd.DataFrame(dataset_dict[SPLIT]).reset_index(names='source_index')
required_columns = {'source_index', 'dataset', 'doc', 'claim', 'label'}
assert required_columns.issubset(dev_df.columns), f'Missing columns: {required_columns - set(dev_df.columns)}'

smoke_df = (
    dev_df.groupby(['dataset', 'label'], group_keys=False)
    .sample(n=SMOKE_PER_DATASET_LABEL, random_state=42)
    .sort_values('source_index')
    .reset_index(drop=True)
)
assert set(smoke_df['source_index']).issubset(set(dev_df['source_index']))
print(f'DEV examples available: {len(dev_df):,}')
print(f'Smoke-test examples selected: {len(smoke_df):,}')
display(smoke_df.groupby(['dataset', 'label']).size().rename('count').reset_index())

DEV examples available: 30,420
Smoke-test examples selected: 44


,dataset,label,count
0,AggreFact-CNN,0,2
1,AggreFact-CNN,1,2
2,AggreFact-XSum,0,2
3,AggreFact-XSum,1,2
4,ClaimVerify,0,2
5,ClaimVerify,1,2
6,ExpertQA,0,2
7,ExpertQA,1,2
8,FactCheck-GPT,0,2
9,FactCheck-GPT,1,2


## 3. Official HalluGuard prompt and XML parser

The prompt content, JSON shape, chat template, and generation settings are taken from the official model card. The parser accepts only the three documented XML categories. Any missing or malformed classification is retained as `PARSE_ERROR` with no binary prediction.

In [6]:
VALID_CLASSES = {
    'GROUNDED': 1,
    'HALLUCINATED_INTRINSIC': 0,
    'HALLUCINATED_EXTRINSIC': 0,
}

def create_official_prompt(document, claim):
    return json.dumps({
        'instructions': [
            'You will be given a document and a claim.',
            "Decide whether the claim is 'GROUNDED', 'HALLUCINATED_INTRINSIC', or 'HALLUCINATED_EXTRINSIC' based ONLY on the document.",
            'Definitions:',
            '  - GROUNDED: The claim is fully supported by the document. All relevant parts are directly verifiable from the document.',
            '  - HALLUCINATED_INTRINSIC: The claim contradicts what the document states or clearly implies.',
            '  - HALLUCINATED_EXTRINSIC: The claim includes information that is not stated or implied in the document and cannot be verified using only the document (it requires external knowledge).',
            'Justification requirements:',
            '  - Your justification MUST be evidence-grounded.',
            '  - Explicitly refer to the relevant parts of the document (by quoting or paraphrasing them).',
            '  - Explain how these parts SUPPORT, CONTRADICT, or FAIL TO SUPPORT the claim.',
            '  - Do NOT use any external knowledge; rely only on the provided document.',
            'Answer format (VERY IMPORTANT):',
            '  - You MUST respond using EXACTLY the following XML structure:',
            '    <answer>',
            '      <classification>CATEGORY</classification>',
            '      <justification>Your reasoning here</justification>',
            '    </answer>',
            '  - CATEGORY must be ONE of: GROUNDED, HALLUCINATED_INTRINSIC, HALLUCINATED_EXTRINSIC.',
            '  - The <justification> must briefly explain your reasoning and cite evidence from the document.',
            '  - Do NOT add any other text before or after the <answer>...</answer> block.',
            '  - Do NOT add any extra tags or attributes.',
        ],
        'document': f"'{document}'",
        'claim': f"'{claim}'",
    }, ensure_ascii=False)

def parse_classification(raw_response):
    match = re.search(r'<classification>\s*([^<]+?)\s*</classification>', raw_response, flags=re.IGNORECASE)
    if not match:
        return 'PARSE_ERROR', None
    predicted_class = match.group(1).strip().upper()
    if predicted_class not in VALID_CLASSES:
        return 'PARSE_ERROR', None
    return predicted_class, VALID_CLASSES[predicted_class]

# Parser-only checks cover all three valid labels without claiming the DEV set contains all three.
for expected_class, expected_binary in VALID_CLASSES.items():
    parsed_class, parsed_binary = parse_classification(f'<answer><classification>{expected_class}</classification></answer>')
    assert (parsed_class, parsed_binary) == (expected_class, expected_binary)
assert parse_classification('unstructured response') == ('PARSE_ERROR', None)
print('Parser checks passed: 3 valid labels map correctly; malformed output remains PARSE_ERROR.')

Parser checks passed: 3 valid labels map correctly; malformed output remains PARSE_ERROR.


## 4. Load HalluGuard and run/resume the smoke test

T4 does not support BF16 efficiently, so FP16 is used for this Kaggle smoke test. `device_map='balanced'` targets both configured GPUs. The official decoding configuration is fixed before evaluation and is not selected from DEV or TEST accuracy.

In [7]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=HF_TOKEN)
max_memory = {gpu_id: '14GiB' for gpu_id in range(torch.cuda.device_count())}
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map='balanced',
    max_memory=max_memory,
    token=HF_TOKEN,
).eval()

GENERATION_KWARGS = {
    'max_new_tokens': 32768,
    'temperature': 0.6,
    'top_p': 0.95,
    'top_k': 20,
    'do_sample': True,
}

def generate_halluguard_response(document, claim):
    prompt = create_official_prompt(document, claim)
    messages = [{'role': 'user', 'content': prompt}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=True)
    inputs = tokenizer([text], return_tensors='pt').to(model.device)
    with torch.inference_mode():
        generated_ids = model.generate(**inputs, **GENERATION_KWARGS)
    output_ids = generated_ids[0][inputs.input_ids.shape[1]:]
    return tokenizer.decode(output_ids, skip_special_tokens=True).strip()

def load_checkpoint(path):
    if not path.exists():
        return pd.DataFrame(columns=['source_index', 'dataset', 'label', 'prediction_3class', 'prediction_binary', 'raw_response', 'status', 'elapsed_seconds'])
    checkpoint = pd.read_csv(path)
    if checkpoint['source_index'].duplicated().any():
        raise ValueError('Checkpoint contains duplicate source_index values; resolve before resuming.')
    return checkpoint

checkpoint_df = load_checkpoint(CHECKPOINT_PATH)
completed_indices = set(checkpoint_df['source_index'].astype(int))
rows_to_run = smoke_df[~smoke_df['source_index'].isin(completed_indices)]
print(f'Resuming: {len(completed_indices)} saved rows; {len(rows_to_run)} rows remaining.')

for gpu_id in range(torch.cuda.device_count()):
    torch.cuda.reset_peak_memory_stats(gpu_id)

run_started = time.perf_counter()
for _, row in rows_to_run.iterrows():
    started = time.perf_counter()
    raw_response = ''
    try:
        raw_response = generate_halluguard_response(row['doc'], row['claim'])
        prediction_3class, prediction_binary = parse_classification(raw_response)
        status = 'OK' if prediction_3class != 'PARSE_ERROR' else 'PARSE_ERROR'
    except RuntimeError as exc:
        if 'out of memory' in str(exc).lower():
            torch.cuda.empty_cache()
            status = 'OOM_ERROR'
        else:
            status = 'RUNTIME_ERROR'
        prediction_3class, prediction_binary = status, None
        raw_response = f'{type(exc).__name__}: {exc}'
    except Exception as exc:
        status = 'RUNTIME_ERROR'
        prediction_3class, prediction_binary = status, None
        raw_response = f'{type(exc).__name__}: {exc}'

    result = {
        'source_index': int(row['source_index']),
        'dataset': row['dataset'],
        'label': int(row['label']),
        'prediction_3class': prediction_3class,
        'prediction_binary': prediction_binary,
        'raw_response': raw_response,
        'status': status,
        'elapsed_seconds': round(time.perf_counter() - started, 3),
    }
    checkpoint_df = pd.concat([checkpoint_df, pd.DataFrame([result])], ignore_index=True)
    checkpoint_df.to_csv(CHECKPOINT_PATH, index=False)  # Save every row for interruption-safe resume.
    print(f"source_index={result['source_index']} status={status} elapsed={result['elapsed_seconds']:.1f}s")
    gc.collect()
    torch.cuda.empty_cache()

wall_clock_seconds = time.perf_counter() - run_started
checkpoint_df.to_csv(OUTPUT_DIR / 'predictions_smoke_final.csv', index=False)
print(f'Run segment complete in {wall_clock_seconds:.1f}s; checkpoint: {CHECKPOINT_PATH}')

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

...

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Resuming: 0 saved rows; 44 rows remaining.
source_index=9 status=OK elapsed=72.4s
source_index=16 status=OK elapsed=101.0s
source_index=31 status=OK elapsed=76.0s
source_index=138 status=OK elapsed=256.9s
source_index=661 status=OK elapsed=71.2s
source_index=707 status=OK elapsed=161.9s
source_index=997 status=OK elapsed=128.2s
source_index=1008 status=OK elapsed=116.7s
source_index=1342 status=OK elapsed=51.2s
source_index=1650 status=OK elapsed=181.7s
source_index=2174 status=OK elapsed=79.6s
source_index=2287 status=OK elapsed=132.8s
source_index=2924 status=OK elapsed=77.3s
source_index=3142 status=OK elapsed=437.2s
source_index=3626 status=OK elapsed=253.9s
source_index=4599 status=OK elapsed=110.7s
source_index=4750 status=OK elapsed=492.8s
source_index=4784 status=OK elapsed=708.0s
source_index=4821 status=OK elapsed=83.1s
source_index=4840 status=OK elapsed=118.0s
source_index=5532 status=OK elapsed=80.5s
source_index=5733 status=OK elapsed=88.2s
source_index=6332 status=OK ela

## 5. Smoke-test validation and concise summary

Balanced Accuracy is computed only over successfully parsed binary predictions and is explicitly labeled as a DEV smoke-test diagnostic. It is not a final benchmark score.

In [8]:
results_df = load_checkpoint(CHECKPOINT_PATH)
expected_indices = set(smoke_df['source_index'].astype(int))
assert set(results_df['source_index'].astype(int)) == expected_indices, 'Processed rows do not match the smoke sample.'
assert not results_df['source_index'].duplicated().any(), 'Duplicate source_index values found.'

valid_df = results_df[results_df['status'] == 'OK'].copy()
parse_error_count = int((results_df['status'] == 'PARSE_ERROR').sum())
failure_count = int((results_df['status'] != 'OK').sum())
if len(valid_df):
    valid_df['prediction_binary'] = valid_df['prediction_binary'].astype(int)
    assert set(valid_df['prediction_binary']).issubset({0, 1})
    smoke_bacc = balanced_accuracy_score(valid_df['label'], valid_df['prediction_binary']) * 100
else:
    smoke_bacc = float('nan')

total_elapsed = float(results_df['elapsed_seconds'].sum())
throughput = len(results_df) / total_elapsed if total_elapsed else float('nan')
peak_gpu_memory_gib = {
    f'gpu_{gpu_id}': round(torch.cuda.max_memory_allocated(gpu_id) / (1024 ** 3), 3)
    for gpu_id in range(torch.cuda.device_count())
}
summary = {
    'stage': 'DEV smoke test only — not a final benchmark result',
    'model': MODEL_NAME,
    'dataset': DATASET_NAME,
    'split': SPLIT,
    'examples': int(len(results_df)),
    'valid_predictions': int(len(valid_df)),
    'parse_errors': parse_error_count,
    'all_failures': failure_count,
    'parse_error_rate_percent': round(parse_error_count / len(results_df) * 100, 2) if len(results_df) else None,
    'runtime_seconds_sum_of_examples': round(total_elapsed, 3),
    'throughput_examples_per_second': round(throughput, 4),
    'balanced_accuracy_percent_dev_smoke_only': round(smoke_bacc, 2) if pd.notna(smoke_bacc) else None,
    'peak_gpu_memory_gib': peak_gpu_memory_gib,
    'generation_config': GENERATION_KWARGS,
}
with open(OUTPUT_DIR / 'smoke_summary.json', 'w', encoding='utf-8') as summary_file:
    json.dump(summary, summary_file, indent=2)

print(json.dumps(summary, indent=2))
print('Smoke-test BAcc is DEV-only engineering validation; do not compare it with the Version 1 test baseline.')

{
  "stage": "DEV smoke test only \u2014 not a final benchmark result",
  "model": "lrsbrgrn/HalluGuard-Qwen3-4B",
  "dataset": "lytang/LLM-AggreFact",
  "split": "dev",
  "examples": 44,
  "valid_predictions": 43,
  "parse_errors": 0,
  "all_failures": 1,
  "parse_error_rate_percent": 0.0,
  "runtime_seconds_sum_of_examples": 6640.457,
  "throughput_examples_per_second": 0.0066,
  "balanced_accuracy_percent_dev_smoke_only": 74.35,
  "peak_gpu_memory_gib": {
    "gpu_0": 12.499,
    "gpu_1": 6.051
  },
  "generation_config": {
    "max_new_tokens": 32768,
    "temperature": 0.6,
    "top_p": 0.95,
    "top_k": 20,
    "do_sample": true
  }
}
Smoke-test BAcc is DEV-only engineering validation; do not compare it with the Version 1 test baseline.
